# Pass stats

Built from scratch (no existing `passes.ipynb` to load) directly on top of
`player_frame_table` / `ball_frame_table` -- same rule as every other stat
module: reads only those two tables, never the raw per-stage caches.

**What counts as a pass, and what doesn't.** Two consecutive possession
segments (player A has the ball, then player B has it) are only scored as a
pass if the ball's tracking during the gap between them looks like a
continuous, on-pitch transfer. If the ball was untracked ("lost") for most
of the gap, or its projected pitch position left the field, the transition
is a **turnover** (shot, dribble knocked away, clearance out of bounds --
then someone else, possibly the other team, eventually gets the ball) and
is excluded from pass stats entirely. This is the direct fix for the case
where a lost dribble or a shot hands the ball to the other team and would
otherwise get counted as a "failed pass" -- it isn't a pass attempt at all.

- **`completed`** = same-team receiver
- **`failed`** = opponent-team receiver (a genuine interception, ball
  control transferred continuously, just to the wrong team)
- **turnover** = ball left continuous tracking during the gap -- not scored
  either way, reported separately

**Known open limitation**, same one flagged in `match_frame_table`: this
still can't perfectly tell a short misplaced pass from a tackle dispossession
that happens to look continuous (no ball-out, no long lost-tracking gap).
Defender-proximity or a ball-velocity-direction check would tighten this
further -- not implemented here.

## Load inputs

Same load pattern as every other stage: everything here is already cached
on disk from `match_frame_table.ipynb`, so this is pure loading, no
recompute. `carrier_cfg.no_candidate_grace_frames` is imported (not
hardcoded) for the same reason it was imported into `build_ball_frame_table`
-- a pass shouldn't be allowed to bridge a gap the carrier assigner itself
already considers "possession lost".

In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import paths
from homography import HomographyConfig
from ball_tracker import CarrierConfig

hom_cfg = HomographyConfig()
carrier_cfg = CarrierConfig()

# TODO: confirm where fps actually lives in your pipeline (DetectionConfig /
# paths.py / a top-level constant) and import it instead of hardcoding this.
FPS = 25

player_frame_table = pd.read_parquet(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
ball_frame_table = pd.read_parquet(paths.BALL_FRAME_TABLE_CACHE_PATH)

print(f"player_frame_table: {player_frame_table.shape}")
print(f"ball_frame_table:   {ball_frame_table.shape}")

c:\Users\user\Desktop\Football_cv_project\football_pressure_analysis\venv_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


player_frame_table: (70396, 10)
ball_frame_table:   (3001, 10)


## Config

`max_gap_frames` reuses `carrier_cfg.no_candidate_grace_frames` so segment
bridging stays in sync with the carrier assigner's own grace period, same
reasoning as the `possession_team` fix in `match_frame_table.ipynb`.

`min_segment_frames` is a genuinely new constant -- nothing upstream to
import it from. It exists to drop possession segments so short they're
almost certainly a carrier-assignment flicker (briefly locks onto a nearby
player for 1-2 frames) rather than real control. Adjust the default if it
turns out to be too aggressive/lenient once you see real segment-length
stats in the sanity checks below.

`ball_lost_gap_fraction_max` and `pitch_out_margin_m` are the fake-pass
guard: if the ball is untracked for more than this fraction of the gap, or
its pitch position leaves the field by more than this margin, the
transition is a turnover, not a pass.

In [2]:
@dataclass
class PassConfig:
    max_gap_frames: int = carrier_cfg.no_candidate_grace_frames

    # NEW constant, no upstream equivalent -- ~0.2s at FPS. Tune after
    # checking the segment-length sanity print below.
    min_segment_frames: int = 5

    # Fraction of the gap-between-segments that must be untracked ("lost")
    # for the gap to be classified as "ball left play" rather than a pass.
    ball_lost_gap_fraction_max: float = 0.5

    # Same margin convention as flag_valid_pitch_coords in
    # match_frame_table.ipynb -- ball pitch position beyond the pitch
    # rectangle by more than this = physically left the field.
    pitch_out_margin_m: float = 5.0


pass_cfg = PassConfig()
print(pass_cfg)

PassConfig(max_gap_frames=8, min_segment_frames=5, ball_lost_gap_fraction_max=0.5, pitch_out_margin_m=5.0)


## Possession segments

Collapse per-frame `carrier_track_id` into contiguous possession segments.
Short carrier-assigner gaps (`<= max_gap_frames`) for the *same* player are
bridged into one segment -- this is what prevents a brief tracking flicker
from turning "player A dribbles for 3 seconds" into a fake pass from A back
to A. Segments shorter than `min_segment_frames` are dropped before they
can become a fake passer or receiver.

In [ ]:
def build_possession_segments(ball_frame_table, player_frame_table, cfg):
    """"One row per possession segment: [track_id, team, start_frame,
    end_frame, n_frames]. Bridges same-player gaps <= cfg.max_gap_frames
    (mirrors the carrier assigner's own grace period) and drops segments
    shorter than cfg.min_segment_frames as tracking noise"""

    team_by_track = (
        player_frame_table[["track_id", "team"]]
        .dropna()
        .drop_duplicates("track_id")
        .set_index("track_id")["team"]
    )

    carrier = ball_frame_table[["frame_idx", "carrier_track_id"]].sort_values("frame_idx")

    segments = []
    cur_id = seg_start = last_seen_frame = None

    def flush(end_frame):
        if cur_id is None:
            return
        n_frames = end_frame - seg_start + 1
        if n_frames >= cfg.min_segment_frames:
            segments.append((cur_id, team_by_track.get(cur_id), seg_start, end_frame, n_frames))

    for f, tid in zip(carrier["frame_idx"], carrier["carrier_track_id"]):
        tid = None if pd.isna(tid) else int(tid)
        if tid is None:
            continue  # gap frame -- bridging handled below when the carrier reappears

        if cur_id is None:
            cur_id, seg_start, last_seen_frame = tid, f, f
            continue

        if tid == cur_id and (f - last_seen_frame) <= cfg.max_gap_frames:
            last_seen_frame = f
            continue

        flush(last_seen_frame)
        cur_id, seg_start, last_seen_frame = tid, f, f

    flush(last_seen_frame)

    seg_df = pd.DataFrame(segments, columns=["track_id", "team", "start_frame", "end_frame", "n_frames"])
    seg_df["team"] = seg_df["team"].astype("Int64")
    return seg_df


seg_df = build_possession_segments(ball_frame_table, player_frame_table, pass_cfg)
print(f"Possession segments -> {len(seg_df)}")
seg_df.head()

Possession segments -> 41


,track_id,team,start_frame,end_frame,n_frames
0,13,1,9,64,56
1,18,1,90,109,20
2,9,1,132,167,36
3,21,1,211,251,41
4,23,1,252,286,35


## Classify transitions: pass vs turnover

For every pair of temporally consecutive segments, look at the ball's
tracking *during the gap between them* (not just who held it before/after):

- if the ball was untracked for more than `ball_lost_gap_fraction_max` of
  the gap, or its pitch position went out of the field by more than
  `pitch_out_margin_m` -> **turnover**, not scored
- otherwise -> genuine pass attempt, scored `completed` (same team) or
  `failed` (different team / interception)

Same-player "transitions" (a segment briefly split despite bridging) are
skipped as a defensive guard, though `max_gap_frames` bridging should
already prevent most of these upstream.

In [ ]:
def classify_transitions(seg_df, ball_frame_table, cfg, hom_cfg):
    """Returns one row per segment-to-segment transition, scored either as
    a pass (outcome = completed/failed) or a turnover (is_turnover = True,
    outcome = None) based on ball tracking continuity during the gap."""
    seg_df = seg_df.sort_values("start_frame").reset_index(drop=True)
    ball = ball_frame_table.set_index("frame_idx").sort_index()

    events = []
    for i in range(len(seg_df) - 1):
        passer, receiver = seg_df.iloc[i], seg_df.iloc[i + 1]
        if passer["track_id"] == receiver["track_id"]:
            continue

        gap_start, gap_end = passer["end_frame"] + 1, receiver["start_frame"] - 1
        gap = ball.loc[gap_start:gap_end] if gap_end >= gap_start else ball.iloc[0:0]
        n_gap = len(gap)

        lost_frac = (gap["ball_source"] == "lost").mean() if n_gap else 0.0

        out_of_bounds = False
        if n_gap:
            x_ok = gap["ball_pitch_x"].between(-cfg.pitch_out_margin_m, hom_cfg.pitch_length + cfg.pitch_out_margin_m)
            y_ok = gap["ball_pitch_y"].between(-cfg.pitch_out_margin_m, hom_cfg.pitch_width + cfg.pitch_out_margin_m)
            tracked = gap["ball_pitch_x"].notna() & gap["ball_pitch_y"].notna()
            out_of_bounds = bool((tracked & ~(x_ok & y_ok)).any())

        is_turnover = (lost_frac > cfg.ball_lost_gap_fraction_max) or out_of_bounds
        outcome = None
        if not is_turnover:
            outcome = "completed" if passer["team"] == receiver["team"] else "failed"

        events.append((
            passer["track_id"], passer["team"], passer["end_frame"],
            receiver["track_id"], receiver["team"], receiver["start_frame"],
            gap_start, gap_end, n_gap, round(lost_frac, 2), out_of_bounds,
            is_turnover, outcome,
        ))

    cols = ["passer_id", "passer_team", "passer_end_frame",
            "receiver_id", "receiver_team", "receiver_start_frame",
            "gap_start", "gap_end", "gap_frames", "ball_lost_frac", "ball_out_of_bounds",
            "is_turnover", "outcome"]
    events_df = pd.DataFrame(events, columns=cols)
    events_df["passer_team"] = events_df["passer_team"].astype("Int64")
    events_df["receiver_team"] = events_df["receiver_team"].astype("Int64")
    return events_df


events_df = classify_transitions(seg_df, ball_frame_table, pass_cfg, hom_cfg)
pass_events = events_df[~events_df["is_turnover"]].copy()
turnovers = events_df[events_df["is_turnover"]].copy()

print(f"Segment transitions     -> {len(events_df)}")
print(f"  genuine pass attempts -> {len(pass_events)}")
print(f"  filtered as turnovers -> {len(turnovers)}  ({len(turnovers)/max(len(events_df),1)*100:.1f}%)")

## Team stats

Attempted / completed / failed, scored to the **passer's** team regardless
of outcome -- an interception is a failed attempt for the passer's team,
not an event credited to the interceptor.

In [ ]:
def team_pass_stats(pass_events):
    stats = (
        pass_events.groupby("passer_team")["outcome"]
        .value_counts()
        .unstack(fill_value=0)
    )
    stats["attempted"] = stats.get("completed", 0) + stats.get("failed", 0)
    stats["completion_pct"] = (stats.get("completed", 0) / stats["attempted"] * 100).round(1)
    stats = stats.reset_index().rename(columns={"passer_team": "team"})
    for col in ["completed", "failed"]:
        if col not in stats.columns:
            stats[col] = 0
    return stats[["team", "attempted", "completed", "failed", "completion_pct"]]


team_stats = team_pass_stats(pass_events)
team_stats

## Top passers by team

Ranked by **completed pass count**, not completion %, so a player with
40/45 completed doesn't get outranked by someone with a flukey 2/2. Adjust
`top_n` as needed.

In [ ]:
def top_passers(pass_events, top_n=5):
    grouped = (
        pass_events.groupby(["passer_team", "passer_id"])["outcome"]
        .value_counts()
        .unstack(fill_value=0)
    )
    for col in ["completed", "failed"]:
        if col not in grouped.columns:
            grouped[col] = 0
    grouped["attempted"] = grouped["completed"] + grouped["failed"]
    grouped["completion_pct"] = (grouped["completed"] / grouped["attempted"] * 100).round(1)
    grouped = grouped.reset_index().rename(columns={"passer_team": "team", "passer_id": "track_id"})
    grouped = grouped.sort_values(["team", "completed"], ascending=[True, False])
    return grouped.groupby("team").head(top_n)[["team", "track_id", "attempted", "completed", "failed", "completion_pct"]]


top_passers_df = top_passers(pass_events, top_n=5)
top_passers_df

## Sanity checks

In [ ]:
print(f"Turnover rate            -> {len(turnovers)/max(len(events_df),1)*100:.1f}% of all segment transitions")
print(f"Pass outcome nulls       -> {pass_events['outcome'].isna().sum()} (should be 0 -- every non-turnover event must resolve)")
print(f"Unknown-team pass events -> {pass_events['passer_team'].isna().sum() + pass_events['receiver_team'].isna().sum()} (referee/unassigned carriers -- check team_by_id upstream if this is high)")

seg_len_avg = seg_df["n_frames"].mean()
print(f"Avg possession segment    -> {seg_len_avg:.1f} frames (~{seg_len_avg/FPS:.2f}s at {FPS}fps)")

print(f"\nTeam totals:\n{team_stats.to_string(index=False)}")

if len(turnovers) > 0.5 * len(events_df):
    print("\n\u26a0\ufe0f  Over half of all transitions are being filtered as turnovers -- ball_lost_gap_fraction_max or pitch_out_margin_m may be too strict, or ball tracking coverage is just low. Cross-check against ball_frame_table's own 'ball position coverage' stat from match_frame_table.ipynb.")

if (team_stats["attempted"] < 5).any():
    print("\n\u26a0\ufe0f  A team has very few pass attempts -- check that team assignment / carrier detection are working for that team before trusting these stats.")

## Cache as parquet

Same load-or-build pattern as every other stage. **Needs
`paths.PASS_EVENTS_CACHE_PATH` added to `paths.py`** -- it doesn't exist yet
since this stage didn't exist before.

In [ ]:
FORCE_REBUILD_PASS_EVENTS = True


def get_or_build_pass_events(force_rebuild=FORCE_REBUILD_PASS_EVENTS):
    events_path = Path(paths.PASS_EVENTS_CACHE_PATH)  # NEW -- add to paths.py

    if events_path.exists() and not force_rebuild:
        print("\u2705 Loaded pass events from cache.")
        return pd.read_parquet(events_path)

    seg = build_possession_segments(ball_frame_table, player_frame_table, pass_cfg)
    events = classify_transitions(seg, ball_frame_table, pass_cfg, hom_cfg)

    events_path.parent.mkdir(parents=True, exist_ok=True)
    events.to_parquet(events_path, index=False)
    print("\U0001F4BE Saved pass events to cache.")
    return events


events_df = get_or_build_pass_events()

## Next steps

`events_df` (all transitions, including turnovers) is the raw cache.
`pass_events` / `turnovers` are the filtered views used for stats above --
any downstream module (pressure, possession-by-zone, etc.) that wants pass
data should re-derive these two filtered views from the cached
`events_df` rather than re-running detection.

Tunables worth revisiting once you've eyeballed a few clips against the
video: `min_segment_frames`, `ball_lost_gap_fraction_max`,
`pitch_out_margin_m`. The sanity-check turnover rate is the fastest signal
for whether these are too strict or too lenient.